# Data Science Project

Students:
- Angel David Abreu Parra - 2121422
- Janika Maria Caroto Correia - 2119122

Teacher:
- Fábio Mendonça

# Project Structure
```
DataScience_IA/
├── DataSet/
│   ├── checkpoint_cleaned_features.pkl     # Saved checkpoint after cleaning & feature engineering
│   ├── checkpoint_part1_complete.pkl       # Final checkpoint (ready for Part 2)
│   ├── cleaned_flight_data.csv             # Cleaned dataset
│   └── flights_sample_3m.csv               # Raw dataset (3M rows)
├── Output_Files/                           # Generated artifacts (plots, reports, logs)
├── Project_Code/
│   ├── JupyterNotebook/
│   │   └── Data_Exploration.ipynb          # Interactive analysis & experiments
│   │
│   └── PythonCode/
│       ├── main.py                         # Main pipeline entry point
│       │
│       ├── DataPreProcessor/
│       │   └── FlightDataCleaner.py        # Data cleaning & preprocessing
│       │
│       ├── EDA/
│       │   └── FlightEDA.py                # Exploratory Data Analysis (EDA)
│       │
│       ├── FeatureEngineering/
│       │   └── FlightFeatureEngineer.py    # Feature generation & transformations
│       │
│       ├── HypothesisTesting/
│       │   └── HypothesisTester.py         # Statistical testing
│       │
│       └── Util/
│           ├── DataLoader.py               # Data loading & train/test split
│           └── DataVisualization.py        # Visualization utilities
├── config.yml                             # Project configuration
├── requirements.txt                       # Python dependencies
├── ARCHITECTURE.md                        # Architecture description
├── IMPLEMENTATION_SUMMARY.md              # Implementation details
└── README.md                              # Project documentation
```

# Flight Delay and Cancellation Analysis (Part 1)

This notebook contains one consolidated and functional Part 1 workflow.
It merges the previous organized narrative with the currently working implementation.

## Notebook Goals

- Build a reliable preprocessing and feature-engineering pipeline.
- Perform exploratory analysis and dimensionality reduction.
- Run hypothesis testing to statistically validate feature relevance.
- Save intermediate and final checkpoints for reproducibility.

## Execution Phases

1. Problem formulation
2. Data loading and cleansing
3. Feature engineering and transformation
4. EDA and dimensionality reduction
5. Hypothesis testing and artifact export

> Tip: set `NROWS = 10000` for a quick run, or `NROWS = None` for the full dataset.


In [1]:
from pathlib import Path
import sys

import pandas as pd

# Resolve project root dynamically to avoid hardcoded absolute paths.
project_root = Path.cwd().resolve()
while project_root != project_root.parent and project_root.name != "DataScience_IA":
    project_root = project_root.parent

if project_root.name != "DataScience_IA":
    raise FileNotFoundError("Could not locate project root folder 'DataScience_IA'.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from Project_Code.PythonCode.DataPreProcessor.FlightDataCleaner import FlightDataCleaner
from Project_Code.PythonCode.FeatureEngeneering.FlightFeatureEngineer import FlightFeatureEngineer
from Project_Code.PythonCode.EDA.FlightEDA import FlightEDA
from Project_Code.PythonCode.Util.DataVisualization import DataVisualization
from Project_Code.PythonCode.Util.DataLoader import DataLoader
from Project_Code.PythonCode.HypothesisTesting.HypothesisTester import HypothesisTester
from Project_Code.PythonCode.Util.ModelSelector import ModelSelector


dataset_path = project_root / "DataSet" / "flights_sample_3m.csv"
output_dir = project_root / "Output_Files"
output_dir.mkdir(parents=True, exist_ok=True)

# DataLoader now auto-downloads the dataset if this local CSV is missing.
# Keep dataset_path as the target location where the CSV should exist/be created.

# Runtime configuration.
NROWS = 10000
TEST_SIZE = 0.2
RANDOM_STATE = 42

print("Project root:", project_root)
print("Dataset path:", dataset_path)
print("Output dir:", output_dir)


Project root: C:\Users\ASUS\Documents\GitHub\DataScienceProj\DataScience_IA
Dataset path: C:\Users\ASUS\Documents\GitHub\DataScienceProj\DataScience_IA\DataSet\flights_sample_3m.csv
Output dir: C:\Users\ASUS\Documents\GitHub\DataScienceProj\DataScience_IA\Output_Files


## Phase 1 - Problem Formulation

This project studies domestic US flight operations (2019-2023) with two predictive targets and one exploratory objective.

### Main Analytical Objectives

1. **Regression task**: predict `ARR_DELAY` (arrival delay in minutes).
2. **Classification task**: predict `DELAY_CLASS` after feature engineering.
3. **Exploratory objective**: identify operational patterns and group-level behavior.

### Leakage-Aware Perspective

A key requirement is to avoid data leakage by excluding post-departure and post-arrival signals during modeling preparation.
This ensures that engineered features represent information realistically available before departure.


In [2]:
phase1_summary = pd.DataFrame(
    {
        "Objective": ["Regression", "Classification", "Pattern Discovery"],
        "Target / Focus": ["ARR_DELAY", "DELAY_CLASS", "EDA + Reduction"],
    }
)
phase1_summary


,Objective,Target / Focus
0,Regression,ARR_DELAY
1,Classification,DELAY_CLASS
2,Pattern Discovery,EDA + Reduction


## Phase 2 - Data Loading and Cleansing

This phase standardizes the raw data preparation pipeline and persists an intermediate checkpoint.

### Steps

1. Load raw data from CSV.
2. Clean raw records (cancelled/diverted filtering, leakage column removal, outlier treatment, null handling).
3. Split cleaned data into train/test sets.
4. Validate resulting cleaned dataset dimensions and quality.


In [3]:
# STEP 1: Load raw data
print("Loading raw dataset...")
loader = DataLoader(str(dataset_path), test_size=TEST_SIZE, random_state=RANDOM_STATE)
df_raw = pd.read_csv(dataset_path, nrows=NROWS)
print(f"Raw data loaded: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")


Loading raw dataset...
Raw data loaded: 10000 rows × 32 columns


In [4]:
# STEP 2: Clean the dataset
cleaner = FlightDataCleaner(df=df_raw)
df_clean = cleaner.load_and_clean(nrows=None, random_state=RANDOM_STATE, balance_method="smote")
df_clean.to_csv("../../DataSet/cleaned_flight_data.csv", index=False)

df_clean = cleaner.classify_target()
print("Clean shape:", df_clean.shape)
print("Numeric missing values after cleaning:", int(df_clean.select_dtypes(include=["number"]).isnull().sum().sum()))
df_clean.head(3)


INICIANDO PROCESSO DE LIMPEZA DE DADOS

1. A usar DataFrame fornecido no construtor...
   ✓ Dataset carregado: 10000 linhas × 32 colunas

2. A remover voos cancelados e desviados...
   ✓ Removidos 270 voos (cancelados/desviados)

3. A remover colunas com data leakage...
   ✓ Removidas 17 colunas com data leakage

4. A remover nulos na variável alvo...
   ✓ Removidas 0 linhas com ARR_DELAY nulo

5. A converter atrasos negativos para zero...
   ✓ 6324 valores negativos de ARR_DELAY convertidos para 0

6. A balancear dataset (método: smote)...
   Distribuição antes do balanceamento: {0: np.int64(6518), 1: np.int64(3212)}
   ! imblearn não instalado. A usar undersampling manual como fallback.
     Instala com: pip install imbalanced-learn
   ✓ Dataset balanceado (manual): 3212 delayed + 3212 on-time

7. A tratar outliers e missing values...
   ✓ DISTANCE: 359 outliers tratados
   ✓ CRS_ELAPSED_TIME: 307 outliers tratados
   ✓ Nenhum missing value encontrado

8. A remover colunas redundante

,FL_DATE,AIRLINE,AIRLINE_CODE,ORIGIN,DEST,CRS_DEP_TIME,CRS_ARR_TIME,ARR_DELAY,CRS_ELAPSED_TIME,DISTANCE,DELAY_CLASS
0,2021-01-09,SkyWest Airlines Inc.,OO,DEN,SUN,1150,1354,11.0,124.0,557.0,On-time
1,2020-02-23,United Air Lines Inc.,UA,MFE,IAH,720,835,0.0,75.0,316.0,On-time
2,2019-02-19,Southwest Airlines Co.,WN,ABQ,SAN,630,730,0.0,120.0,628.0,On-time


In [5]:
# STEP 3: Split cleaned data into train/test sets
loader.data = df_clean
data_train, data_test, target_train, target_test = loader.split_data(target_column='ARR_DELAY')

print("Train shape:", data_train.shape)
print("Test shape:", data_test.shape)
print("Target train shape:", target_train.shape if target_train is not None else None)
print("Target test shape:", target_test.shape if target_test is not None else None)



Data split completed: 5139 train × 1285 test
Train shape: (5139, 10)
Test shape: (1285, 10)
Target train shape: (5139,)
Target test shape: (1285,)


## Phase 2 - Exploratory Data Analysis and Dimensionality Reduction

This section combines analytical diagnostics with visual diagnostics.
In this phase we will conduct EDA in which we gather insights on the data. We aim to conduct descriptive statistics through statistical techniques and deriving graphs for visualization. As well as performing dimension reduction to get a general understanding of the data.

This phase will therefore be divided into 2 categories:

### Analytical diagnostics

- Distribution and descriptive summaries
- Variable ranges and correlation structure
- Data quality checks

### Dimensionality reduction

- PCA for linear structure
- UMAP (or t-SNE fallback) for non-linear structure


In [6]:
# STEP 7: Run full EDA routine
eda = FlightEDA(df_clean, target_col="ARR_DELAY", output_dir=output_dir, group_col="DELAY_CLASS")
eda_report = eda.perform_eda()

print("Missing values total:", int(eda_report["quality"]["missing_values"].sum()))
print("Duplicate rows:", int(eda_report["quality"]["duplicate_count"]))
print("Top outlier counts:")
print(eda_report["quality"]["outlier_count"].head(5))


INICIANDO ANALISE EXPLORATORIA (EDA)
INICIANDO EDA ANALITICA

ESTATISTICAS DESCRITIVAS
Resumo geral:
       CRS_DEP_TIME  CRS_ARR_TIME  CRS_ELAPSED_TIME     DISTANCE    ARR_DELAY
count   6424.000000   6424.000000       6424.000000  6424.000000  6424.000000
mean    1342.808531   1502.213730        132.775053   717.698599    19.316936
std      484.618361    516.789233         53.282328   417.425159    55.174007
min        7.000000      1.000000         33.000000    61.000000     0.000000
25%      930.000000   1117.000000         91.000000   391.750000     0.000000
50%     1336.000000   1534.000000        126.000000   668.000000     0.500000
75%     1735.000000   1927.000000        164.000000   966.000000    17.000000
max     2359.000000   2359.000000        301.000000  2016.000000  1185.000000

Resumo por grupo (DELAY_CLASS):
            CRS_DEP_TIME                                                  \
                   count         mean         std   min      25%     50%   
DELAY_CLASS 

In [7]:
# STEP 10: Additional visual diagnostics
viz = DataVisualization(df_clean, output_dir=output_dir)
viz.plot_heatmap_top_correlations(top_n=10)

Generating top 10 correlations heatmap...
[OK] Heatmap saved: viz_heatmap_top_correlations.png


In [8]:
# STEP 8: PCA
df_clean.head(3)
pca_components = eda.run_pca(n_components=2, explained_variance_threshold=0.8)

print("PCA type:", type(pca_components))
print("PCA shape:", pca_components.shape)



EXECUTANDO PCA

✓ PCA executado e visualizado

PCA type: <class 'numpy.ndarray'>
PCA shape: (6424, 2)


In [9]:
# STEP 9: UMAP/t-SNE
umap_components = eda.run_umap_or_tsne(n_components=2, use_umap=True)

print("UMAP/t-SNE type:", type(umap_components))
print("UMAP/t-SNE shape:", umap_components.shape)



EXECUTANDO UMAP / t-SNE

Aplicando UMAP...

✓ UMAP executado e visualizado

UMAP/t-SNE type: <class 'numpy.ndarray'>
UMAP/t-SNE shape: (6424, 2)


Graphs to show delay class

In [10]:
if "DELAY_CLASS" in df_clean.columns:
    focus_cols = ["DISTANCE", "CRS_ELAPSED_TIME", "PLANNED_SPEED_MPM", "ARR_DELAY"]
    viz.plot_grouped_feature_distributions(
        columns=focus_cols,
        group_col="DELAY_CLASS",
        filename="viz_grouped_distributions_delay_class.png",
    )
    viz.plot_grouped_boxplots(
        columns=focus_cols,
        group_col="DELAY_CLASS",
        filename="viz_grouped_boxplots_delay_class.png",
    )

Generating grouped histograms for 3 columns...
[OK] Grouped histograms saved: viz_grouped_distributions_delay_class.png
Generating grouped boxplots for 3 columns...
[OK] Grouped boxplots saved: viz_grouped_boxplots_delay_class.png


Description of outputs of EDA:\

Below shows a list of output files derived from EDA and that can be found in the output directory for future reference.

In [11]:
generated_eda_files = sorted([p.name for p in output_dir.glob("eda_*.png")])
print("Generated EDA files:")
for name in generated_eda_files:
    print("-", name)

Generated EDA files:
- eda_boxplots.png
- eda_correlation_matrix.png
- eda_distributions.png
- eda_grouped_boxplots.png
- eda_grouped_distributions.png
- eda_pca_2d.png
- eda_target_distribution.png
- eda_umap_2d.png


## Phase 3 - Feature Engineering and Transformation

This phase converts cleaned operational fields into model-ready features.

### What is created in this step

- Temporal behavior features
- Route and interaction features
- Delay classification target (`DELAY_CLASS`)
- Encoded categorical columns
- Normalized numeric predictors


In [12]:
# Remove leakage columns
df_removed = cleaner.remove_data_leak_cols()

# STEP 3: Generate features
engineer = FlightFeatureEngineer(df_removed)
df_features = engineer.generate_features()

# STEP 4: Encode categorical features
df_features = engineer.encode_categorical()

# STEP 5: Normalize numeric features
df_features = engineer.normalize_features(method="both")

print("Features shape:", df_features.shape)
print("Total columns:", len(df_features.columns))
print("DELAY_CLASS present:", "DELAY_CLASS" in df_features.columns)
df_features.head(3)


INICIANDO FEATURE ENGINEERING

1. Processando features temporais...
   ✓ 10 features temporais criadas

2. Processando features de rota...
   ✓ 3 features de rota criadas

3. Processando features de categorização...
   ✓ 3 features de categorização criadas

4. Processando features de interação...
   ✓ 3 features de interação criadas

FEATURE ENGINEERING CONCLUÍDO!
Total de features novas: 20+
Dimensão do dataset: 6424 linhas × 30 colunas

Codificando variáveis categóricas...

   [One-Hot Encoding] colunas nominais: ['AIRLINE', 'AIRLINE_CODE', 'TIME_PERIOD']
   Justificação: variáveis nominais sem ordem natural — OHE evita
   que o modelo assuma relações ordinais falsas entre categorias.
   ✓ 39 colunas OHE criadas a partir de 3 variáveis

   [Label Encoding] colunas ordinais/alta cardinalidade: ['DISTANCE_CAT', 'DURATION_CAT', 'SPEED_CAT', 'DELAY_CLASS', 'ROUTE']
   Justificação: variáveis com ordem natural (Short<Medium<Long) ou
   alta cardinalidade (ROUTE) onde OHE criaria demasiada

,ORIGIN,DEST,CRS_DEP_TIME,CRS_ARR_TIME,ARR_DELAY,CRS_ELAPSED_TIME,DISTANCE,DELAY_CLASS,MONTH,DAY_OF_WEEK,...,AIRLINE_CODE_OH,AIRLINE_CODE_OO,AIRLINE_CODE_QX,AIRLINE_CODE_UA,AIRLINE_CODE_WN,AIRLINE_CODE_YV,AIRLINE_CODE_YX,TIME_PERIOD_Afternoon,TIME_PERIOD_Morning,TIME_PERIOD_Night
0,DEN,SUN,-0.397887,-0.286820,11.0,-0.164703,-0.385006,1,-1.590207,1.012769,...,-0.203312,2.801515,-0.079156,-0.289259,-0.497128,-0.138554,-0.217434,-0.769973,1.283312,-0.57735
1,MFE,IAH,-1.285253,-1.291176,0.0,-1.084404,-0.962400,1,-1.292622,1.513155,...,-0.203312,-0.356950,-0.079156,3.457110,-0.497128,-0.138554,-0.217434,-0.769973,1.283312,-0.57735
2,ABQ,SAN,-1.470980,-1.494369,0.0,-0.239780,-0.214902,1,-1.292622,-0.988778,...,-0.203312,-0.356950,-0.079156,-0.289259,2.011553,-0.138554,-0.217434,-0.769973,1.283312,-0.57735


In [13]:
# STEP 6: Intermediate checkpoint
loader.data = df_features
checkpoint_clean_path = dataset_path.parent / "checkpoint_cleaned_features.pkl"
loader.save_checkpoint(str(checkpoint_clean_path))

print("Checkpoint created:", checkpoint_clean_path.exists())
print("Checkpoint path:", checkpoint_clean_path)


Checkpoint salvo: C:\Users\ASUS\Documents\GitHub\DataScienceProj\DataScience_IA\DataSet\checkpoint_cleaned_features.pkl
Checkpoint created: True
Checkpoint path: C:\Users\ASUS\Documents\GitHub\DataScienceProj\DataScience_IA\DataSet\checkpoint_cleaned_features.pkl


## Phase 3 - Model Selection

In [14]:
print("Running model selection and validation strategy assessment...")
selector = ModelSelector(
    data=df_features,
    target_col="DELAY_CLASS",
    output_dir=output_dir,
    random_state=RANDOM_STATE,
    cv_folds=5,
)
selection_report = selector.run_model_selection()

for name, df_result in selection_report.items():
    out_csv = output_dir / f"model_selection_{name}.csv"
    df_result.to_csv(out_csv, index=False)
    print(f"Saved: {out_csv.name}")


Running model selection and validation strategy assessment...

STARTING PHASE 3 — MODEL SELECTION

PHASE 3 — CANDIDATE MODEL EXAMINATION
Problem type : Multi-class classification (3 classes)
Target       : DELAY_CLASS — On-time | Short delay | Long delay
Dataset size : Large (100k+ rows) — favour scalable models
Features     : Mix of numeric and encoded categorical

  [✓ Suitable] Baseline (majority class)
    Reference baseline.
    Any useful model must outperform random/majority guessing.
    Provides a lower bound on acceptable performance..

  [✓ Suitable] Logistic Regression
    Linear probabilistic classifier.
    Interpretable coefficients allow understanding which features drive delay predictions.
    Fast training.
    Suitable as a strong linear baseline.
    Requires feature scaling (already applied).
    Limitation: assumes linear decision boundaries..

  [✓ Suitable] k-Nearest Neighbors (kNN)
    Non-parametric, instance-based learner.
    No training phase — predictions 

## Phase 5 - Hypothesis Testing and Final Checkpoint

In [15]:
print("Running statistical tests with formal H0/H1 formulation...")

# Main battery: normality, ANOVA, Kruskal-Wallis, Levene, pairwise t-tests.
hypothesis_tester = HypothesisTester(
    data=df_features,
    target_col="DELAY_CLASS",
    verbose=True,
)
summary_report = hypothesis_tester.generate_summary_report()

# Specific hypothesis: time of day impact on delays.
print("   -> Hypothesis: Time of Day Impact on Delays")
if "TIME_PERIOD" in df_features.columns and "DEP_DELAY" in df_features.columns:
    tester_time = HypothesisTester(data=df_features, target_col="TIME_PERIOD", verbose=True)
    summary_report["time_period_impact"] = tester_time.test_time_of_day_impact()
else:
    print("   (TIME_PERIOD or DEP_DELAY not found - skipping time-of-day test)")

# Specific hypothesis: airline systematic delays.
print("   -> Hypothesis: Airline Systematic Delays")
if "AIRLINE" in df_features.columns and "ARR_DELAY" in df_features.columns:
    tester_airline = HypothesisTester(data=df_features, target_col="AIRLINE", verbose=True)
    summary_report["airline_systemic_delays"] = tester_airline.test_airline_delays()
else:
    print("   (AIRLINE or ARR_DELAY not found - skipping airline test)")

print("Exporting statistical reports...")
for test_name, results_df in summary_report.items():
    if results_df is None or results_df.empty:
        continue
    out_csv = output_dir / f"hypothesis_test_{test_name}.csv"
    results_df.to_csv(out_csv, index=False)
    print("Report saved:", out_csv.name)


Running statistical tests with formal H0/H1 formulation...

HYPOTHESIS TESTING — FULL BATTERY

───────────────────────────────────────────────────────
  Test: Shapiro-Wilk Normality Test
  H0 (null hypothesis)       : The feature follows a normal distribution.
  H1 (alternative hypothesis): The feature does not follow a normal distribution.
  Significance level (α)     : 0.05
───────────────────────────────────────────────────────
  → REJECT H0 [CRS_DEP_TIME]: p=0.000000 < α=0.05 — significant result.
  → REJECT H0 [CRS_ARR_TIME]: p=0.000000 < α=0.05 — significant result.
  → REJECT H0 [ARR_DELAY]: p=0.000000 < α=0.05 — significant result.
  → REJECT H0 [CRS_ELAPSED_TIME]: p=0.000000 < α=0.05 — significant result.
  → REJECT H0 [DISTANCE]: p=0.000000 < α=0.05 — significant result.
  → REJECT H0 [MONTH]: p=0.000000 < α=0.05 — significant result.
  → REJECT H0 [DAY_OF_WEEK]: p=0.000000 < α=0.05 — significant result.
  → REJECT H0 [DAY_OF_YEAR]: p=0.000000 < α=0.05 — significant result.
 

In [16]:
# Final Checkpoint
print("Saving final checkpoint...")
loader.data = df_features
checkpoint_final_path = dataset_path.parent / "checkpoint_part1_complete.pkl"
loader.save_checkpoint(str(checkpoint_final_path))
print("Final Checkpoint saved:", checkpoint_final_path)


Saving final checkpoint...
Checkpoint salvo: C:\Users\ASUS\Documents\GitHub\DataScienceProj\DataScience_IA\DataSet\checkpoint_part1_complete.pkl
Final Checkpoint saved: C:\Users\ASUS\Documents\GitHub\DataScienceProj\DataScience_IA\DataSet\checkpoint_part1_complete.pkl
